# 🇬🇭 Practical Labs — Ghana Fintech Data Pipeline
## Hands-On: Build It Yourself

---

Welcome to the lab notebook! Here you write and run real code. By the end, you'll have a working ETL pipeline processing mock MoMo transaction data using AWS services — all locally, no AWS account needed.

> 💡 **How this works**: We use `moto` — a Python library that intercepts `boto3` (the AWS SDK) calls and runs them against a local in-memory mock. Your code looks identical to production code. To go live, you'd remove the moto decorators and point to a real AWS account.

**Estimated time:** ~1.5 hours  
**Prerequisites:** Run the theory notebook first (`01_theory_data_ingestion_pipeline.ipynb`)

---

## 📋 Table of Contents

- [Setup](#setup) — Install packages and configure environment
- [Lab 1 — Extract](#lab1) — Generate data and upload to mock S3
- [Lab 2 — Transform](#lab2) — Clean, normalize, and enrich transactions
- [Lab 3 — Load](#lab3) — Store in DynamoDB and query results
- [Lab 4 — Automate & Monitor](#lab4) — Full pipeline with logging and alerts

---

<a id='setup'></a>
# ⚙️ Setup
### Install dependencies and configure the mock AWS environment

---

In [ ]:
# ============================================================
# Cell 0.1 — Install required packages
# WHAT: Install boto3 (AWS SDK), moto (AWS mock), pandas, faker
# WHY:  moto lets us run AWS code without a real AWS account
# ============================================================

# Uncomment the line below if running for the first time
# !pip install boto3 pandas 'moto[s3,dynamodb,sns]' faker --quiet

print("✅ If you see no errors after running the install, you're ready to go!")
print("   If you haven't installed yet, uncomment the !pip install line above and run again.")

In [ ]:
# ============================================================
# Cell 0.2 — Import all dependencies
# WHAT: Load every library we'll use across all four labs
# WHY:  Importing everything upfront prevents mid-lab surprises
# ============================================================

import json
import uuid
import random
import logging
import re
from datetime import datetime, timedelta, timezone
from decimal import Decimal

import boto3
import pandas as pd
from moto import mock_aws

# Configure logging so we see our pipeline messages
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)
logger = logging.getLogger("fintech_pipeline")

# Fix random seed so your results match the workshop
random.seed(42)

# Pipeline configuration constants
RAW_BUCKET      = "fintech-pipeline-raw"
PROCESSED_BUCKET = "fintech-pipeline-processed"
DYNAMODB_TABLE  = "momo_transactions"
SNS_TOPIC_NAME  = "pipeline-alerts"
AWS_REGION      = "us-east-1"  # moto default region

print("✅ All imports successful!")
print(f"   Raw bucket      : {RAW_BUCKET}")
print(f"   Processed bucket: {PROCESSED_BUCKET}")
print(f"   DynamoDB table  : {DYNAMODB_TABLE}")
print(f"   SNS topic       : {SNS_TOPIC_NAME}")

In [ ]:
# ============================================================
# Cell 0.3 — Transaction data generator
# WHAT: Create 500 realistic but intentionally messy MoMo transactions
# WHY:  Real pipeline code must handle dirty data — we simulate that here
# ============================================================

# Ghanaian names pool
GH_FIRST_NAMES = [
    "Kwame","Ama","Kofi","Akosua","Yaw","Abena","Kweku","Adwoa","Kwabena","Afia",
    "Kojo","Esi","Fiifi","Efua","Kwesi","Akua","Nana","Adzo","Ato","Araba",
    "Mensah","Adjoa","Kobina","Ewurabena","Serwaa","Osei","Maame","Asante","Abenaa",
    "Ibrahim","Fatima","Mohammed","Alima","Abdul","Rahinatu","Sumaila","Zenabu",
    "Emmanuel","Grace","Daniel","Patience","Joseph","Abigail","Samuel","Rejoice"
]

GH_LAST_NAMES = [
    "Mensah","Asante","Boateng","Owusu","Darko","Acheampong","Amponsah","Frimpong",
    "Adusei","Kyei","Osei","Opoku","Appiah","Antwi","Adjei","Amoah","Bonsu",
    "Asare","Ntim","Afriyie","Quaye","Tetteh","Nortey","Laryea","Ankrah","Okine",
    "Lomotey","Tsatsu","Agbemafle","Klu","Mahama","Bawumia","Alhassan","Sulemana",
    "Issahaku","Yakubu","Fusheini","Aidoo","Forson","Gyasi","Anim","Wiredu"
]

REGIONS = [
    "Greater Accra","Ashanti","Western","Eastern","Northern",
    "Volta","Central","Upper East","Upper West","Bono"
]

# Dirty region variants — simulating inconsistent data from different agent apps
DIRTY_REGION_MAP = {
    "Greater Accra": ["Greater Accra","Gt. Accra","greater accra","GREATER ACCRA","Accra Region"],
    "Ashanti":       ["Ashanti","Ashanti Region","ashanti","ASHANTI","Kumasi Region"],
    "Western":       ["Western","Western Region","western","WESTERN"],
    "Eastern":       ["Eastern","Eastern Region","eastern","EASTERN"],
    "Northern":      ["Northern","Northern Region","northern","NORTHERN"],
    "Volta":         ["Volta","Volta Region","volta","VOLTA"],
    "Central":       ["Central","Central Region","central","CENTRAL"],
    "Upper East":    ["Upper East","Upper East Region","upper east","UPPER EAST"],
    "Upper West":    ["Upper West","Upper West Region","upper west","UPPER WEST"],
    "Bono":          ["Bono","Bono Region","bono","BONO","Brong-Ahafo"]
}

REGION_AGENT_PREFIX = {
    "Greater Accra":"ACC","Ashanti":"KSI","Western":"TAK","Eastern":"KFR",
    "Northern":"TME","Volta":"HHO","Central":"CAP","Upper East":"BOL",
    "Upper West":"WA","Bono":"SUY"
}

TXN_TYPES  = ["P2P","CASH_IN","CASH_OUT","MERCHANT_PAYMENT","BILL_PAYMENT","AIRTIME_PURCHASE"]
STATUSES   = ["SUCCESS","FAILED","PENDING","REVERSED"]
STATUS_W   = [0.75, 0.10, 0.08, 0.07]
NETWORKS   = ["MTN","Vodafone","AirtelTigo","Glo"]
CHANNELS   = ["USSD","APP","WEB","POS","AGENT"]


def _random_phone():
    """Return a phone number in one of several messy real-world formats."""
    prefixes   = ["020","024","026","027","028","050","054","055","057","059"]
    local_pre  = random.choice(prefixes)
    local_tail = "".join([str(random.randint(0,9)) for _ in range(7)])
    local      = local_pre + local_tail          # e.g. 0244123456 — 10 digits

    fmt = random.randint(0, 5)
    if fmt == 0: return local                                          # 0244XXXXXXX
    if fmt == 1: return f"+233{local[1:]}"                           # +233244XXXXXXX
    if fmt == 2: return f"233{local[1:]}"                            # 233244XXXXXXX
    if fmt == 3: return f"+233 {local[1:4]} {local[4:7]} {local[7:]}" # spaced
    if fmt == 4: return f"{local[:4]}-{local[4:7]}-{local[7:]}"     # dashed
    return local


def _random_timestamp():
    """Higher transaction volume during Ghana daytime (UTC 06:00–22:00)."""
    base = datetime(2026, 4, 20, 0, 0, 0)
    hour_weights = [1,1,1,1,2,3,5,7,8,9,9,8,8,7,7,8,9,9,8,7,5,4,3,2]
    hour   = random.choices(range(24), weights=hour_weights)[0]
    minute = random.randint(0, 59)
    second = random.randint(0, 59)
    ts     = base + timedelta(hours=hour, minutes=minute, seconds=second)
    return ts.strftime("%Y-%m-%dT%H:%M:%SZ")


def generate_transactions(n=500):
    """
    Generate n MoMo transactions with realistic Ghanaian data.
    Intentional dirty data included:
      - ~15 duplicate txn_ids
      - mixed phone number formats
      - inconsistent region names
      - ~1% missing statuses (None)
      - ~2% negative amounts
    """
    transactions = []
    generated_ids = []

    for i in range(n + 15):  # overshoot so we can embed duplicates and trim
        region     = random.choice(REGIONS)
        dirty_rgn  = random.choice(DIRTY_REGION_MAP[region])
        agent_pfx  = REGION_AGENT_PREFIX[region]
        agent_id   = f"AGT-{agent_pfx}-{random.randint(1,99):03d}"
        txn_type   = random.choices(TXN_TYPES, weights=[25,20,20,15,12,8])[0]
        status     = random.choices(STATUSES, weights=STATUS_W)[0]

        # Inject duplicate IDs for about 3% of records
        if generated_ids and random.random() < 0.03:
            txn_id = random.choice(generated_ids)
        else:
            txn_id = f"TXN-{uuid.uuid4().hex[:12].upper()}"
        generated_ids.append(txn_id)

        # Occasionally negative amount (dirty data)
        amount = round(random.uniform(-200, -1), 2) if random.random() < 0.02 \
                 else round(random.uniform(1, 5000), 2)

        # Occasionally null status (dirty data)
        final_status = None if random.random() < 0.01 else status

        has_receiver = txn_type not in ["BILL_PAYMENT", "AIRTIME_PURCHASE"]

        txn = {
            "txn_id":        txn_id,
            "timestamp":     _random_timestamp(),
            "sender_phone":  _random_phone(),
            "receiver_phone": _random_phone() if has_receiver else None,
            "sender_name":   f"{random.choice(GH_FIRST_NAMES)} {random.choice(GH_LAST_NAMES)}",
            "receiver_name": f"{random.choice(GH_FIRST_NAMES)} {random.choice(GH_LAST_NAMES)}" if has_receiver else None,
            "amount_ghs":    amount,
            "type":          txn_type,
            "status":        final_status,
            "region":        dirty_rgn,
            "agent_id":      agent_id,
            "reference":     f"REF-{random.randint(100000,999999)}",
            "channel":       random.choice(CHANNELS),
            "fee_ghs":       round(amount * 0.01, 2) if amount > 0 else 0.0,
            "network":       random.choice(NETWORKS)
        }
        transactions.append(txn)

    return transactions[:n]


# Generate our dataset now
raw_transactions = generate_transactions(500)

print(f"✅ Generated {len(raw_transactions)} transactions")
print(f"   Sample record:")
print(json.dumps(raw_transactions[0], indent=4))

<a id='lab1'></a>
# Lab 1 — Extract: Ingesting Transaction Data
### *Getting raw data into our pipeline's entry point: S3*

---

**What we're doing:** Simulate the first step of our ETL pipeline — a batch of MoMo transactions arrives and we upload it to the raw S3 bucket.

In a real system, this upload might be triggered by:
- A scheduled Lambda that calls the MTN MoMo API at midnight
- An SFTP listener that detects a new file from a GhIPSS drop
- A webhook from Hubtel when a merchant's daily summary is ready

For this lab, we'll upload our generated transactions directly.

> ⚠️ **Moto note**: Every cell in Labs 1–3 uses the `@mock_aws` decorator or is called from within a mocked context. In Lab 4, we wrap everything cleanly. For now, we'll start and stop the mock manually so you can run cells individually.

---

In [ ]:
# ============================================================
# Cell 1.1 — Inspect the raw data before uploading
# WHAT: Load transactions into a DataFrame to see what we're dealing with
# WHY:  Always inspect your data before processing — know what's dirty
# ============================================================

raw_df = pd.DataFrame(raw_transactions)

print("=" * 60)
print("RAW TRANSACTION DATA — OVERVIEW")
print("=" * 60)
print(f"\nTotal records : {len(raw_df)}")
print(f"Columns       : {list(raw_df.columns)}")
print()
print(raw_df.dtypes)
print()
print("First 3 records:")
raw_df.head(3)

In [ ]:
# ============================================================
# Cell 1.2 — Spot the dirty data before we clean it
# WHAT: Count known data quality issues in the raw dataset
# WHY:  Gives us a baseline to verify our transform step later
# ============================================================

print("=" * 60)
print("DATA QUALITY AUDIT — RAW TRANSACTIONS")
print("=" * 60)

# Duplicate transaction IDs
dup_count = raw_df.duplicated(subset=["txn_id"]).sum()
print(f"\n🔴 Duplicate txn_ids          : {dup_count}")

# Missing / null statuses
null_status = raw_df["status"].isna().sum()
print(f"🔴 Missing statuses (null)     : {null_status}")

# Negative amounts
neg_amounts = (raw_df["amount_ghs"] < 0).sum()
print(f"🔴 Negative amounts            : {neg_amounts}")

# Mixed phone formats (quick check — count those NOT starting with 0)
non_local_phones = raw_df["sender_phone"].str.startswith("0").value_counts()
print(f"🟡 sender_phone format mix     : {dict(non_local_phones)}")

# Inconsistent region names (should be exactly 10 canonical values)
unique_regions = raw_df["region"].unique()
print(f"🟡 Unique region values        : {len(unique_regions)} (expected: 10)")
print(f"   Sample region values        : {sorted(unique_regions[:8])}")

# Transaction type distribution
print(f"\n📊 Transaction type breakdown:")
print(raw_df["type"].value_counts().to_string())

In [ ]:
# ============================================================
# Cell 1.3 — Start mock AWS and create S3 buckets
# WHAT: Initialize the moto mock and create our two S3 buckets
# WHY:  moto intercepts boto3 calls — no real AWS credentials needed
# ============================================================

# Start the mock context (we'll keep it alive across lab cells)
mock = mock_aws()
mock.start()

# Create a boto3 S3 client pointing to the mock
s3_client = boto3.client("s3", region_name=AWS_REGION)

# Create the raw data bucket
s3_client.create_bucket(Bucket=RAW_BUCKET)
print(f"✅ Created S3 bucket: {RAW_BUCKET}")

# Create the processed data bucket
s3_client.create_bucket(Bucket=PROCESSED_BUCKET)
print(f"✅ Created S3 bucket: {PROCESSED_BUCKET}")

# Verify both buckets exist
response = s3_client.list_buckets()
print(f"\n📦 All buckets in mock AWS:")
for bucket in response["Buckets"]:
    print(f"   - {bucket['Name']}")

In [ ]:
# ============================================================
# Cell 1.4 — Upload raw transactions to S3
# WHAT: Serialize transactions to JSON and PUT them in the raw S3 bucket
# WHY:  S3 is our pipeline's entry point — all processing starts here
# ============================================================

# Define the S3 key (path) using date-based partitioning
# This is a best practice — makes it easy to find and replay specific days
raw_s3_key = "transactions/2026/04/20/batch_001.json"

# Serialize the transaction list to a JSON string
raw_json_payload = json.dumps(raw_transactions, indent=2)
payload_size_kb  = len(raw_json_payload.encode()) / 1024

print(f"📤 Uploading {len(raw_transactions)} transactions...")
print(f"   Payload size : {payload_size_kb:.1f} KB")
print(f"   Destination  : s3://{RAW_BUCKET}/{raw_s3_key}")

s3_client.put_object(
    Bucket=RAW_BUCKET,
    Key=raw_s3_key,
    Body=raw_json_payload.encode("utf-8"),
    ContentType="application/json"
)

print(f"\n✅ Upload complete!")

In [ ]:
# ============================================================
# Cell 1.5 — Verify the upload by listing and reading back
# WHAT: List bucket contents and read back the uploaded JSON
# WHY:  Always verify writes — silent failures are common in distributed systems
# ============================================================

# List objects in the raw bucket
list_response = s3_client.list_objects_v2(Bucket=RAW_BUCKET)

print("📦 Objects in raw bucket:")
print(f"   {'Key':<60} {'Size (bytes)':<15}")
print(f"   {'-'*60} {'-'*15}")
for obj in list_response.get("Contents", []):
    print(f"   {obj['Key']:<60} {obj['Size']:<15,}")

# Read the file back from S3 to confirm the data is correct
get_response  = s3_client.get_object(Bucket=RAW_BUCKET, Key=raw_s3_key)
retrieved_raw = json.loads(get_response["Body"].read().decode("utf-8"))

print(f"\n✅ Read-back verification:")
print(f"   Records uploaded : {len(raw_transactions)}")
print(f"   Records retrieved: {len(retrieved_raw)}")
print(f"   Match            : {len(raw_transactions) == len(retrieved_raw)}")

## ✅ Lab 1 Checkpoint

**Question:** How many transactions did we upload to S3? Print the exact count and payload size.

```python
# Your answer here:
print(f"Uploaded: {len(raw_transactions)} transactions")
print(f"Payload : {len(json.dumps(raw_transactions).encode()) / 1024:.1f} KB")
```

> 💡 **What happens in production**: Instead of Python generating the data, a Lambda function would call the MTN MoMo API or read from an SFTP drop and upload the result to this same S3 path. The rest of the pipeline is identical.

---

<a id='lab2'></a>
# Lab 2 — Transform: Cleaning & Enriching
### *Turning messy raw data into trustworthy clean data*

---

**Cleaning checklist for our MoMo transaction data:**

- [ ] Normalize phone numbers → `+233XXXXXXXXX` format
- [ ] Detect and remove duplicate `txn_id` records
- [ ] Standardize region names ("Gt. Accra" → "Greater Accra")
- [ ] Validate transaction types against allowed list
- [ ] Fill missing statuses with `"UNKNOWN"`
- [ ] Drop records with negative amounts
- [ ] Calculate agent commission per transaction type
- [ ] Add `processed_at` metadata timestamp

---

In [ ]:
# ============================================================
# Cell 2.1 — Read raw data from S3 into a DataFrame
# WHAT: Retrieve the uploaded JSON from S3 and load into pandas
# WHY:  pandas gives us powerful vectorized operations for cleaning
# ============================================================

get_response = s3_client.get_object(Bucket=RAW_BUCKET, Key=raw_s3_key)
raw_data     = json.loads(get_response["Body"].read().decode("utf-8"))

# Load into DataFrame
df = pd.DataFrame(raw_data)

print("=" * 60)
print("RAW DATA LOADED FROM S3")
print("=" * 60)
print(f"\nShape: {df.shape[0]} rows × {df.shape[1]} columns")
print()
df.info()
print()
df[["txn_id","sender_phone","amount_ghs","type","status","region"]].head(5)

In [ ]:
# ============================================================
# Cell 2.2 — Phone number normalization function
# WHAT: Convert any Ghana phone format to canonical +233XXXXXXXXX
# WHY:  Inconsistent phone formats make matching/deduplication impossible
# ============================================================

def normalize_phone(raw_number):
    """
    Normalize a Ghana phone number to +233XXXXXXXXX format.

    Handles inputs like:
      0244123456        → +233244123456
      +233244123456     → +233244123456  (already correct)
      233244123456      → +233244123456  (missing +)
      +233 244 123 456  → +233244123456  (remove spaces)
      0244-123-456      → +233244123456  (remove dashes)

    Returns None for unrecognizable numbers.
    """
    if not raw_number or not isinstance(raw_number, str):
        return None

    # Strip all whitespace and dashes
    cleaned = re.sub(r"[\s\-]", "", raw_number.strip())

    # Case 1: Already in correct international format → return as-is
    if re.match(r"^\+233\d{9}$", cleaned):
        return cleaned

    # Case 2: International format without + (e.g. 233244123456)
    if re.match(r"^233\d{9}$", cleaned):
        return "+" + cleaned

    # Case 3: Local format starting with 0 (e.g. 0244123456 — 10 digits)
    if re.match(r"^0\d{9}$", cleaned):
        return "+233" + cleaned[1:]  # replace leading 0 with +233

    # Unrecognized format — return None so we can flag it
    return None


# --- Test the function with edge cases ---
test_cases = [
    ("0244123456",        "+233244123456"),
    ("+233244123456",     "+233244123456"),
    ("233244123456",      "+233244123456"),
    ("+233 244 123 456",  "+233244123456"),
    ("0244-123-456",      "+233244123456"),
    ("INVALID",           None),
    (None,                None),
]

print("Phone normalization test:")
print(f"  {'Input':<25} {'Expected':<20} {'Got':<20} {'Pass?'}")
print(f"  {'-'*25} {'-'*20} {'-'*20} {'-'*5}")
all_pass = True
for raw, expected in test_cases:
    result = normalize_phone(raw)
    passed = result == expected
    all_pass = all_pass and passed
    mark = "✅" if passed else "❌"
    print(f"  {str(raw):<25} {str(expected):<20} {str(result):<20} {mark}")

print(f"\n{'✅ All tests passed!' if all_pass else '❌ Some tests failed — check the function.'}")

In [ ]:
# ============================================================
# Cell 2.3 — Apply phone normalization to the DataFrame
# WHAT: Run normalize_phone() on both sender and receiver columns
# WHY:  Consistent phone numbers enable accurate matching across datasets
# ============================================================

cleaned_df = df.copy()  # always work on a copy — preserve the raw data

cleaned_df["sender_phone"]   = cleaned_df["sender_phone"].apply(normalize_phone)
cleaned_df["receiver_phone"] = cleaned_df["receiver_phone"].apply(normalize_phone)

# Count how many phones couldn't be normalized
bad_sender   = cleaned_df["sender_phone"].isna().sum()
bad_receiver = cleaned_df["receiver_phone"].isna().sum()

print(f"✅ Phone normalization applied")
print(f"   Sender phones not parseable   : {bad_sender}")
print(f"   Receiver phones not parseable : {bad_receiver} (includes BILL_PAYMENT & AIRTIME_PURCHASE — expected)")
print(f"\nSample normalized sender phones:")
print(cleaned_df["sender_phone"].head(10).to_string())

In [ ]:
# ============================================================
# Cell 2.4 — Detect and remove duplicate transactions
# WHAT: Find records with the same txn_id, keep first, log duplicates
# WHY:  Idempotency — processing a duplicate twice could double-credit agents
# ============================================================

# Identify all duplicate rows (keeping the first occurrence)
is_duplicate      = cleaned_df.duplicated(subset=["txn_id"], keep="first")
duplicate_txns    = cleaned_df[is_duplicate].copy()
duplicate_count   = len(duplicate_txns)

print(f"🔍 Duplicate detection results:")
print(f"   Total records        : {len(cleaned_df)}")
print(f"   Duplicate txn_ids    : {duplicate_count}")
print(f"   Unique transactions  : {len(cleaned_df) - duplicate_count}")

if duplicate_count > 0:
    print(f"\n   Duplicate txn_ids found:")
    for txn_id in duplicate_txns["txn_id"].values:
        logger.warning(f"Duplicate transaction flagged and removed: {txn_id}")
    print(duplicate_txns[["txn_id","amount_ghs","type","status"]].head())

# Remove duplicates — keep first occurrence
cleaned_df = cleaned_df.drop_duplicates(subset=["txn_id"], keep="first").reset_index(drop=True)

print(f"\n✅ After deduplication: {len(cleaned_df)} records remain")

In [ ]:
# ============================================================
# Cell 2.5 — Standardize region names
# WHAT: Map all region name variants to the 10 canonical Ghana region names
# WHY:  Inconsistent regions break any grouping/aggregation by region
# ============================================================

# Build a flat lookup: every variant → canonical name
REGION_NORMALIZER = {}
for canonical, variants in DIRTY_REGION_MAP.items():
    for variant in variants:
        REGION_NORMALIZER[variant.lower()] = canonical  # case-insensitive

def standardize_region(raw_region):
    """Map a raw region string to its canonical form."""
    if not isinstance(raw_region, str):
        return "UNKNOWN"
    return REGION_NORMALIZER.get(raw_region.lower(), f"UNKNOWN: {raw_region}")


# Track what we're about to change
before_regions = cleaned_df["region"].value_counts()

cleaned_df["region"] = cleaned_df["region"].apply(standardize_region)

after_regions = cleaned_df["region"].value_counts()

print("Region standardization:")
print(f"  Unique values BEFORE: {len(before_regions)}")
print(f"  Unique values AFTER : {len(after_regions)}")
print()
print("Distribution after standardization:")
print(after_regions.to_string())

In [ ]:
# ============================================================
# Cell 2.6 — Validate transaction types and fill missing statuses
# WHAT: Ensure type column only contains known values; fill null statuses
# WHY:  Unknown types and null statuses break downstream reporting logic
# ============================================================

VALID_TYPES    = {"P2P","CASH_IN","CASH_OUT","MERCHANT_PAYMENT","BILL_PAYMENT","AIRTIME_PURCHASE"}
VALID_STATUSES = {"SUCCESS","FAILED","PENDING","REVERSED"}

# Flag unknown transaction types
unknown_types = ~cleaned_df["type"].isin(VALID_TYPES)
if unknown_types.any():
    logger.warning(f"{unknown_types.sum()} records have unknown transaction types")
    cleaned_df.loc[unknown_types, "type"] = "UNKNOWN"
print(f"✅ Transaction types validated — {unknown_types.sum()} unknowns flagged")

# Fill missing/null statuses
null_statuses = cleaned_df["status"].isna().sum()
cleaned_df["status"] = cleaned_df["status"].fillna("UNKNOWN")

# Validate remaining statuses
invalid_statuses = ~cleaned_df["status"].isin(VALID_STATUSES | {"UNKNOWN"})
if invalid_statuses.any():
    cleaned_df.loc[invalid_statuses, "status"] = "UNKNOWN"

print(f"✅ {null_statuses} null statuses set to UNKNOWN")
print(f"\nStatus distribution after cleaning:")
print(cleaned_df["status"].value_counts().to_string())

In [ ]:
# ============================================================
# Cell 2.7 — Remove negative amounts and round to 2dp
# WHAT: Drop rows with amount_ghs < 0; round all amounts to 2 decimal places
# WHY:  Negative amounts indicate corrupt records; rounding prevents float errors
# ============================================================

neg_mask = cleaned_df["amount_ghs"] < 0
neg_count = neg_mask.sum()

print(f"Negative amounts found: {neg_count}")
if neg_count > 0:
    print("  Records being dropped:")
    print(cleaned_df[neg_mask][["txn_id","amount_ghs","type"]].to_string())
    for txn_id in cleaned_df[neg_mask]["txn_id"]:
        logger.warning(f"Dropping record with negative amount: {txn_id}")

cleaned_df = cleaned_df[~neg_mask].reset_index(drop=True)

# Round amounts and fees to exactly 2 decimal places
cleaned_df["amount_ghs"] = cleaned_df["amount_ghs"].round(2)
cleaned_df["fee_ghs"]    = cleaned_df["fee_ghs"].round(2)

print(f"\n✅ After removing negatives: {len(cleaned_df)} records remain")

In [ ]:
# ============================================================
# Cell 2.8 — Calculate agent commission
# WHAT: Add agent_commission_ghs column based on transaction type
# WHY:  Commission is how MoMo agents earn income — must be accurate for reconciliation
# ============================================================

# Commission rates by transaction type (industry-approximate values for Ghana)
COMMISSION_RATES = {
    "CASH_IN":          0.005,   # 0.5% of transaction amount
    "CASH_OUT":         0.010,   # 1.0% of transaction amount
    "P2P":              0.002,   # 0.2% — agents rarely involved in P2P
    "MERCHANT_PAYMENT": 0.008,   # 0.8%
    "BILL_PAYMENT":     0.005,   # 0.5%
    "AIRTIME_PURCHASE": 0.003,   # 0.3%
}

def calculate_commission(row):
    """Return agent commission for this transaction."""
    rate = COMMISSION_RATES.get(row["type"], 0.0)
    # Only pay commission on successful transactions
    if row["status"] != "SUCCESS":
        return 0.00
    return round(row["amount_ghs"] * rate, 2)

cleaned_df["agent_commission_ghs"] = cleaned_df.apply(calculate_commission, axis=1)

print("✅ Agent commission calculated")
print(f"\nCommission summary by transaction type:")
commission_summary = cleaned_df.groupby("type")["agent_commission_ghs"].agg(["sum","mean","count"])
commission_summary.columns = ["Total Commission (GHS)","Avg Commission (GHS)","Count"]
commission_summary = commission_summary.round(2)
print(commission_summary.to_string())
print(f"\nTotal agent commissions payable: GHS {cleaned_df['agent_commission_ghs'].sum():,.2f}")

In [ ]:
# ============================================================
# Cell 2.9 — Add metadata and show before/after comparison
# WHAT: Add processed_at timestamp; display cleaning summary
# WHY:  Metadata enables auditability — you know when and how data was processed
# ============================================================

# Add processing metadata
cleaned_df["processed_at"]  = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
cleaned_df["source_batch"]  = "batch_001.json"
cleaned_df["pipeline_ver"]  = "1.0.0"

# --- Before/After Summary ---
print("=" * 60)
print("TRANSFORMATION SUMMARY")
print("=" * 60)
print(f"  Records in (raw)         : {len(raw_df)}")
print(f"  Duplicates removed       : {duplicate_count}")
print(f"  Negative amounts dropped : {neg_count}")
print(f"  Records out (clean)      : {len(cleaned_df)}")
print(f"  Retention rate           : {len(cleaned_df)/len(raw_df)*100:.1f}%")
print()
print(f"  Null statuses fixed      : {null_status}")
print(f"  Region variants resolved : Yes (all → canonical names)")
print(f"  Phone numbers normalized : Yes (+233XXXXXXXXX format)")
print(f"  Commission calculated     : Yes (per transaction type)")
print()
print("Sample cleaned record:")
cleaned_df[["txn_id","sender_phone","amount_ghs","type","status","region","agent_commission_ghs"]].head(3)

In [ ]:
# ============================================================
# Cell 2.10 — Upload cleaned data to the processed S3 bucket
# WHAT: Save cleaned DataFrame as CSV to S3 processed zone
# WHY:  Processed zone is the source of truth for all downstream analytics
# ============================================================

processed_s3_key = "transactions/2026/04/20/batch_001_clean.csv"

# Serialize cleaned DataFrame to CSV in memory
csv_payload = cleaned_df.to_csv(index=False).encode("utf-8")

print(f"📤 Uploading cleaned data...")
print(f"   Records          : {len(cleaned_df)}")
print(f"   Columns          : {len(cleaned_df.columns)}")
print(f"   Payload size     : {len(csv_payload)/1024:.1f} KB")
print(f"   Destination      : s3://{PROCESSED_BUCKET}/{processed_s3_key}")

s3_client.put_object(
    Bucket=PROCESSED_BUCKET,
    Key=processed_s3_key,
    Body=csv_payload,
    ContentType="text/csv"
)

print(f"\n✅ Cleaned data uploaded to processed bucket!")

## ✅ Lab 2 Checkpoint

Run the cell below to answer:
1. How many duplicate transactions were found?
2. What percentage of transactions were CASH_OUT?

---

In [ ]:
# ✅ Checkpoint answers
cashout_pct = (cleaned_df["type"] == "CASH_OUT").sum() / len(cleaned_df) * 100
print(f"Duplicates found       : {duplicate_count}")
print(f"CASH_OUT percentage    : {cashout_pct:.1f}%")
print(f"Total clean records    : {len(cleaned_df)}")

<a id='lab3'></a>
# Lab 3 — Load: Storing & Querying
### *Loading clean transactions into DynamoDB and running queries*

---

**What we're doing:** Load the cleaned transactions into a DynamoDB table and run four queries that mirror real customer-service and reporting needs at a Ghanaian fintech company.

---

In [ ]:
# ============================================================
# Cell 3.1 — Create the DynamoDB table
# WHAT: Create momo_transactions table with txn_id (PK) and timestamp (SK)
# WHY:  PK+SK design enables both single-item lookup and time-range queries
# ============================================================

dynamodb = boto3.resource("dynamodb", region_name=AWS_REGION)

table = dynamodb.create_table(
    TableName=DYNAMODB_TABLE,
    KeySchema=[
        {"AttributeName": "txn_id",    "KeyType": "HASH"},   # Partition key
        {"AttributeName": "timestamp", "KeyType": "RANGE"},  # Sort key
    ],
    AttributeDefinitions=[
        {"AttributeName": "txn_id",    "AttributeType": "S"},  # String
        {"AttributeName": "timestamp", "AttributeType": "S"},  # String (ISO format sorts correctly)
    ],
    BillingMode="PAY_PER_REQUEST"  # no capacity planning needed at our scale
)

print(f"✅ DynamoDB table created: {DYNAMODB_TABLE}")
print(f"   Partition key : txn_id    (String)")
print(f"   Sort key      : timestamp (String)")
print(f"   Billing mode  : PAY_PER_REQUEST")

In [ ]:
# ============================================================
# Cell 3.2 — Batch write all transactions to DynamoDB
# WHAT: Write all cleaned records using DynamoDB batch_writer
# WHY:  batch_writer auto-handles the 25-item batch limit and retries
# ============================================================

# DynamoDB cannot store Python float — must use Decimal
def to_dynamodb_item(row_dict):
    """Convert a dict so it's safe for DynamoDB (float→Decimal, None→removed)."""
    item = {}
    for key, value in row_dict.items():
        if value is None or (isinstance(value, float) and pd.isna(value)):
            continue  # DynamoDB doesn't accept None — skip the attribute
        elif isinstance(value, float):
            item[key] = Decimal(str(round(value, 2)))
        else:
            item[key] = value
    return item


records = cleaned_df.to_dict(orient="records")
loaded_count = 0
error_count  = 0

print(f"⏳ Loading {len(records)} records into DynamoDB...")

with table.batch_writer() as writer:
    for record in records:
        try:
            item = to_dynamodb_item(record)
            writer.put_item(Item=item)
            loaded_count += 1
        except Exception as e:
            logger.error(f"Failed to load txn {record.get('txn_id')}: {e}")
            error_count += 1

print(f"\n✅ Batch load complete!")
print(f"   Successfully loaded : {loaded_count}")
print(f"   Errors              : {error_count}")
logger.info(f"Loaded {loaded_count} clean transactions into DynamoDB table '{DYNAMODB_TABLE}'")

In [ ]:
# ============================================================
# Cell 3.3 — Query 1: Get a specific transaction by ID
# WHAT: Use DynamoDB get_item to retrieve a single transaction
# WHY:  Customer service queries — "what happened to transaction TXN-XYZ?"
# ============================================================

# Pick the first transaction's ID to look up
lookup_txn_id  = cleaned_df.iloc[0]["txn_id"]
lookup_ts      = cleaned_df.iloc[0]["timestamp"]

response = table.get_item(
    Key={
        "txn_id":    lookup_txn_id,
        "timestamp": lookup_ts
    }
)

item = response.get("Item", {})

print(f"🔍 Query 1 — Single transaction lookup")
print(f"   txn_id    : {lookup_txn_id}")
print(f"   timestamp : {lookup_ts}")
print()
if item:
    print(f"   ✅ Found:")
    for k in ["txn_id","amount_ghs","type","status","region","sender_phone","agent_id"]:
        print(f"     {k:<25}: {item.get(k)}")
else:
    print("   ❌ Not found")

In [ ]:
# ============================================================
# Cell 3.4 — Query 2: Find all FAILED transactions
# WHAT: Scan the table for status == FAILED
# WHY:  Failed transactions need investigation and possible reversal
# ============================================================

from boto3.dynamodb.conditions import Attr

scan_response = table.scan(
    FilterExpression=Attr("status").eq("FAILED")
)

failed_items  = scan_response["Items"]
failed_df     = pd.DataFrame(failed_items)

# Convert Decimal back to float for display
if not failed_df.empty:
    failed_df["amount_ghs"] = failed_df["amount_ghs"].apply(float)

print(f"🔍 Query 2 — Failed transactions")
print(f"   Total FAILED records: {len(failed_df)}")
print(f"   Total value at risk : GHS {failed_df['amount_ghs'].sum():,.2f}")
print()
if not failed_df.empty:
    print(failed_df[["txn_id","amount_ghs","type","region","channel"]].head(10).to_string(index=False))

In [ ]:
# ============================================================
# Cell 3.5 — Query 3: Total transaction volume per region
# WHAT: Scan all records and aggregate total GHS volume by region
# WHY:  Regional reporting — which regions drive the most MoMo volume?
# ============================================================

# Use the in-memory cleaned_df for this aggregation (faster than a full DynamoDB scan)
region_summary = (
    cleaned_df
    .groupby("region", as_index=False)
    .agg(
        total_volume_ghs=("amount_ghs",  "sum"),
        txn_count       =("txn_id",       "count"),
        avg_txn_ghs     =("amount_ghs",  "mean"),
        success_count   =("status",       lambda x: (x == "SUCCESS").sum())
    )
    .sort_values("total_volume_ghs", ascending=False)
    .reset_index(drop=True)
)

region_summary["total_volume_ghs"] = region_summary["total_volume_ghs"].round(2)
region_summary["avg_txn_ghs"]      = region_summary["avg_txn_ghs"].round(2)
region_summary["success_rate"]     = (
    region_summary["success_count"] / region_summary["txn_count"] * 100
).round(1).astype(str) + "%"

print("🔍 Query 3 — Transaction volume by region")
print()
print(region_summary[["region","txn_count","total_volume_ghs","avg_txn_ghs","success_rate"]].to_string(index=False))
print(f"\n🇬🇭 Highest volume region: {region_summary.iloc[0]['region']} "
      f"(GHS {region_summary.iloc[0]['total_volume_ghs']:,.2f})")

In [ ]:
# ============================================================
# Cell 3.6 — Query 4: Top 10 agents by transaction count
# WHAT: Rank agents by number of transactions processed
# WHY:  Identifies top performers for bonus programs and fraud risk profiling
# ============================================================

agent_summary = (
    cleaned_df
    .groupby("agent_id", as_index=False)
    .agg(
        txn_count          =("txn_id",               "count"),
        total_volume_ghs   =("amount_ghs",           "sum"),
        total_commission   =("agent_commission_ghs", "sum"),
        region             =("region",               "first")
    )
    .sort_values("txn_count", ascending=False)
    .head(10)
    .reset_index(drop=True)
)

agent_summary["total_volume_ghs"] = agent_summary["total_volume_ghs"].round(2)
agent_summary["total_commission"] = agent_summary["total_commission"].round(2)
agent_summary.index = agent_summary.index + 1  # rank from 1

print("🔍 Query 4 — Top 10 Agents by Transaction Count")
print()
print(agent_summary[["agent_id","region","txn_count","total_volume_ghs","total_commission"]]
      .rename(columns={
          "txn_count":"# Txns",
          "total_volume_ghs":"Volume (GHS)",
          "total_commission":"Commission (GHS)"
      }).to_string())

In [ ]:
# ============================================================
# Cell 3.7 — SNS alert: notify team about failed transactions
# WHAT: Create an SNS topic and publish a failure alert message
# WHY:  Automated alerting means the team knows about problems in seconds
# ============================================================

sns_client = boto3.client("sns", region_name=AWS_REGION)

# Create the alert topic
topic_response = sns_client.create_topic(Name=SNS_TOPIC_NAME)
topic_arn      = topic_response["TopicArn"]

# Subscribe an email endpoint (mock — no real email sent in moto)
sns_client.subscribe(
    TopicArn=topic_arn,
    Protocol="email",
    Endpoint="fintech-team@mymomo.com.gh"
)

# Build and publish the alert message
failed_count       = (cleaned_df["status"] == "FAILED").sum()
failed_volume_ghs  = cleaned_df[cleaned_df["status"] == "FAILED"]["amount_ghs"].sum()

alert_message = (
    f"⚠️  PIPELINE ALERT — batch_001.json\n"
    f"Date          : 2026-04-20\n"
    f"Failed txns   : {failed_count}\n"
    f"Value at risk : GHS {failed_volume_ghs:,.2f}\n"
    f"Action needed : Review failed transactions in DynamoDB table '{DYNAMODB_TABLE}'\n"
    f"Pipeline run  : {datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ')}"
)

sns_client.publish(
    TopicArn=topic_arn,
    Subject="[MoMo Pipeline] Failed Transactions Detected",
    Message=alert_message
)

print(f"✅ SNS topic created    : {topic_arn}")
print(f"✅ Alert published to   : fintech-team@mymomo.com.gh")
print(f"\nAlert content:")
print("-" * 50)
print(alert_message)
print("-" * 50)

## ✅ Lab 3 Checkpoint

**Question:** Which region had the highest total transaction volume (GHS)?

```python
# Your answer (already computed above):
print(f"Highest volume region: {region_summary.iloc[0]['region']}")
print(f"Total volume: GHS {region_summary.iloc[0]['total_volume_ghs']:,.2f}")
```

---

<a id='lab4'></a>
# Lab 4 — Automate & Monitor
### *Tying everything into one automated, observable pipeline*

---

**What we're doing:** Wrap the entire Extract → Transform → Load workflow into a single `run_pipeline()` function with:
- Structured logging
- Error handling and SNS failure alerts
- A Lambda-ready handler structure
- CloudWatch log simulation
- EventBridge scheduling explanation

---

In [ ]:
# ============================================================
# Cell 4.1 — The complete run_pipeline() function
# WHAT: Single function that chains Extract → Transform → Load end-to-end
# WHY:  One callable entry point makes it easy to schedule, test, and rerun
# ============================================================

def run_pipeline(batch_name, s3_client, dynamodb_table, sns_topic_arn):
    """
    Full ETL pipeline for a single MoMo transaction batch.

    Args:
        batch_name     : e.g. "batch_001.json"
        s3_client      : boto3 S3 client
        dynamodb_table : boto3 DynamoDB Table resource
        sns_topic_arn  : ARN of the SNS alert topic

    Returns:
        dict with pipeline run statistics
    """
    run_start  = datetime.now(timezone.utc)
    stats      = {"extracted": 0, "duplicates": 0, "negatives": 0,
                  "loaded": 0, "errors": 0, "failed_txns": 0}

    logger.info(f"Pipeline started — batch: {batch_name}")

    try:
        # ── EXTRACT ──────────────────────────────────────────────
        raw_key = f"transactions/2026/04/20/{batch_name}"
        response = s3_client.get_object(Bucket=RAW_BUCKET, Key=raw_key)
        raw_data = json.loads(response["Body"].read().decode("utf-8"))
        stats["extracted"] = len(raw_data)
        logger.info(f"Extracted {stats['extracted']} transactions from s3://{RAW_BUCKET}/{raw_key}")

        work_df = pd.DataFrame(raw_data)

        # ── TRANSFORM ────────────────────────────────────────────
        # 1. Normalize phones
        work_df["sender_phone"]   = work_df["sender_phone"].apply(normalize_phone)
        work_df["receiver_phone"] = work_df["receiver_phone"].apply(normalize_phone)
        logger.info("Phone numbers normalized")

        # 2. Deduplicate
        dups = work_df.duplicated(subset=["txn_id"], keep="first").sum()
        stats["duplicates"] = int(dups)
        work_df = work_df.drop_duplicates(subset=["txn_id"], keep="first").reset_index(drop=True)
        if dups:
            logger.warning(f"{dups} duplicate transactions flagged and removed")

        # 3. Standardize regions
        work_df["region"] = work_df["region"].apply(standardize_region)
        logger.info("Region names standardized")

        # 4. Fill missing statuses
        null_s = work_df["status"].isna().sum()
        work_df["status"] = work_df["status"].fillna("UNKNOWN")
        if null_s:
            logger.warning(f"{null_s} null statuses set to UNKNOWN")

        # 5. Drop negatives
        negs = (work_df["amount_ghs"] < 0).sum()
        stats["negatives"] = int(negs)
        work_df = work_df[work_df["amount_ghs"] >= 0].reset_index(drop=True)
        if negs:
            logger.warning(f"{negs} records with negative amounts dropped")

        # 6. Round amounts
        work_df["amount_ghs"] = work_df["amount_ghs"].round(2)
        work_df["fee_ghs"]    = work_df["fee_ghs"].round(2)

        # 7. Commission
        work_df["agent_commission_ghs"] = work_df.apply(calculate_commission, axis=1)

        # 8. Metadata
        work_df["processed_at"] = run_start.strftime("%Y-%m-%dT%H:%M:%SZ")
        work_df["source_batch"] = batch_name
        work_df["pipeline_ver"] = "1.0.0"

        logger.info(f"Transform complete — {len(work_df)} clean records ready")

        # ── LOAD to S3 processed ──────────────────────────────────
        proc_key = f"transactions/2026/04/20/{batch_name.replace('.json','_clean.csv')}"
        s3_client.put_object(
            Bucket=PROCESSED_BUCKET,
            Key=proc_key,
            Body=work_df.to_csv(index=False).encode("utf-8"),
            ContentType="text/csv"
        )
        logger.info(f"Cleaned data saved to s3://{PROCESSED_BUCKET}/{proc_key}")

        # ── LOAD to DynamoDB ──────────────────────────────────────
        records   = work_df.to_dict(orient="records")
        loaded    = 0
        err_count = 0

        with dynamodb_table.batch_writer() as writer:
            for record in records:
                try:
                    writer.put_item(Item=to_dynamodb_item(record))
                    loaded += 1
                except Exception as e:
                    logger.error(f"DynamoDB write failed for {record.get('txn_id')}: {e}")
                    err_count += 1

        stats["loaded"]     = loaded
        stats["errors"]     = err_count
        stats["failed_txns"] = int((work_df["status"] == "FAILED").sum())

        logger.info(f"Loaded {loaded} transactions into DynamoDB '{DYNAMODB_TABLE}'")

        # ── ALERT if failures detected ────────────────────────────
        if stats["failed_txns"] > 0:
            failed_vol = work_df[work_df["status"]=="FAILED"]["amount_ghs"].sum()
            msg = (
                f"⚠️  {stats['failed_txns']} failed transactions detected in {batch_name}\n"
                f"Value at risk: GHS {failed_vol:,.2f}\n"
                f"Action: Review DynamoDB table '{DYNAMODB_TABLE}' — filter status=FAILED"
            )
            sns_client_local = boto3.client("sns", region_name=AWS_REGION)
            sns_client_local.publish(
                TopicArn=sns_topic_arn,
                Subject="[MoMo Pipeline] Failed Transactions Detected",
                Message=msg
            )
            logger.warning(f"SNS alert sent — {stats['failed_txns']} failed transactions")

    except Exception as pipeline_error:
        logger.error(f"Pipeline FAILED: {pipeline_error}")
        # Alert on pipeline-level failure
        try:
            sns_client_local = boto3.client("sns", region_name=AWS_REGION)
            sns_client_local.publish(
                TopicArn=sns_topic_arn,
                Subject="[MoMo Pipeline] CRITICAL FAILURE",
                Message=f"❌ Pipeline crashed processing {batch_name}\nError: {pipeline_error}"
            )
        except Exception:
            pass  # don't let SNS failure mask the original error
        raise

    run_duration = (datetime.now(timezone.utc) - run_start).total_seconds()
    stats["duration_seconds"] = round(run_duration, 2)
    logger.info(f"Pipeline completed in {run_duration:.2f}s — stats: {stats}")
    return stats


print("✅ run_pipeline() function defined — ready to execute in Cell 4.2")

In [ ]:
# ============================================================
# Cell 4.2 — Execute the full pipeline end-to-end
# WHAT: Run run_pipeline() with our mock AWS resources
# WHY:  Validate the entire ETL flow works as a single unit
# ============================================================

print("=" * 60)
print("RUNNING FULL PIPELINE")
print("=" * 60)
print()

pipeline_stats = run_pipeline(
    batch_name     = "batch_001.json",
    s3_client      = s3_client,
    dynamodb_table = table,
    sns_topic_arn  = topic_arn
)

print()
print("=" * 60)
print("PIPELINE RUN SUMMARY")
print("=" * 60)
for stat_key, stat_val in pipeline_stats.items():
    print(f"  {stat_key:<25}: {stat_val}")

In [ ]:
# ============================================================
# Cell 4.3 — Lambda handler structure (not deployed — reference only)
# WHAT: Show how run_pipeline() would be wrapped for AWS Lambda
# WHY:  This is the exact structure you'd deploy to Lambda in production
# ============================================================

lambda_handler_code = '''
import json
import boto3
import logging

logger  = logging.getLogger()
logger.setLevel(logging.INFO)

# Initialise AWS clients once outside the handler (reused across warm invocations)
s3_client  = boto3.client("s3")
dynamodb   = boto3.resource("dynamodb")
table      = dynamodb.Table("momo_transactions")
SNS_ARN    = "arn:aws:sns:us-east-1:123456789012:pipeline-alerts"


def lambda_handler(event, context):
    """
    AWS Lambda entry point.

    Can be triggered by:
      - EventBridge schedule (cron) → event contains schedule metadata
      - S3 event notification      → event contains bucket/key info
      - Manual invocation          → event contains batch_name key
    """
    logger.info(f"Lambda invoked. Event: {json.dumps(event)}")

    # Determine batch name from the trigger event
    if "Records" in event:  # S3 trigger
        s3_key     = event["Records"][0]["s3"]["object"]["key"]
        batch_name = s3_key.split("/")[-1]
    else:  # Schedule trigger or manual
        batch_name = event.get("batch_name", "batch_001.json")

    stats = run_pipeline(batch_name, s3_client, table, SNS_ARN)

    return {
        "statusCode": 200,
        "body": json.dumps(stats)
    }
'''

print("Lambda handler structure (production reference):")
print(lambda_handler_code)

## ⏰ EventBridge Scheduling — How to Run This Every Night

In production, you'd create an **EventBridge rule** to schedule your Lambda:

```yaml
# AWS CloudFormation / SAM template snippet
Resources:
  MoMoPipelineSchedule:
    Type: AWS::Events::Rule
    Properties:
      Name: momo-pipeline-nightly
      Description: Run MoMo ETL pipeline every night at 10pm UTC (midnight Ghana time)
      ScheduleExpression: "cron(0 22 * * ? *)"   # 10pm UTC daily
      State: ENABLED
      Targets:
        - Arn: !GetAtt MoMoPipelineLambda.Arn
          Id: MoMoPipelineTarget
          Input: '{"batch_name": "batch_001.json"}'
```

**Cron expression breakdown:** `cron(0 22 * * ? *)`

| Field | Value | Meaning |
|-------|-------|---------|
| Minutes | `0` | At minute 0 |
| Hours | `22` | At 10pm UTC (= midnight Ghana time) |
| Day of month | `*` | Every day |
| Month | `*` | Every month |
| Day of week | `?` | Any (ignored when day-of-month is `*`) |
| Year | `*` | Every year |

> 🇬🇭 **Ghana time note**: Ghana runs on GMT (UTC+0) year-round — no daylight saving. So 10pm UTC = 10pm Ghana time. Clean and simple!

---

## 🔍 CloudWatch Monitoring — What Your Dashboard Would Show

```
┌──────────────────────────────────────────────────────────────────────────┐
│          CloudWatch Dashboard: MoMo Pipeline Health                      │
│          Period: 2026-04-20  |  Last run: 22:01 UTC                      │
├────────────────┬───────────────┬────────────────┬────────────────────────┤
│ Lambda         │ Error Rate    │ Duration       │ Records Processed      │
│ Invocations    │               │                │                        │
│                │               │                │                        │
│     1          │    0.0%       │   2.3 sec      │    488 loaded          │
│  ✅ Healthy    │  ✅ Normal    │  ✅ Fast       │  ✅ DynamoDB           │
├────────────────┴───────────────┴────────────────┴────────────────────────┤
│ Recent Log Events (CloudWatch Logs > /aws/lambda/momo-etl-pipeline)      │
│                                                                          │
│ 22:01:00 INFO  Pipeline started — batch: batch_001.json                  │
│ 22:01:00 INFO  Extracted 500 transactions from s3://fintech-raw/...      │
│ 22:01:01 INFO  Phone numbers normalized                                  │
│ 22:01:01 WARN  12 duplicate transactions flagged and removed             │
│ 22:01:01 INFO  Region names standardized                                 │
│ 22:01:01 WARN  3 null statuses set to UNKNOWN                            │
│ 22:01:01 WARN  2 records with negative amounts dropped                   │
│ 22:01:01 INFO  Transform complete — 486 clean records ready              │
│ 22:01:02 INFO  Cleaned data saved to s3://fintech-processed/...          │
│ 22:01:03 INFO  Loaded 486 transactions into DynamoDB 'momo_transactions' │
│ 22:01:03 WARN  SNS alert sent — 47 failed transactions                   │
│ 22:01:03 INFO  Pipeline completed in 2.31s — stats: {...}               │
└──────────────────────────────────────────────────────────────────────────┘
```

> 💡 **CloudWatch Alarm**: You can set an alarm: *"If Lambda error rate > 5% for 2 consecutive periods, send an SNS alert."* This means the team gets woken up only when something is truly broken, not for expected warnings.

---

In [ ]:
# ============================================================
# Cell 4.4 — Clean up: stop the moto mock
# WHAT: Stop the moto AWS mock context we started in Lab 1
# WHY:  Good practice to clean up mock resources after use
# ============================================================

mock.stop()
print("✅ Moto AWS mock stopped. All in-memory resources cleared.")
print("   In production, your S3 buckets and DynamoDB table would persist.")

---

# 🏁 Congratulations! You Built a Production-Grade Pipeline

---

You just built what fintech companies in Accra run in production — every night, for hundreds of thousands of transactions.

## What You Built Today

```
┌─────────────────────────────────────────────────────────────────────────┐
│              YOUR COMPLETED PIPELINE ARCHITECTURE                       │
│                                                                         │
│  ┌──────────┐    ┌──────────────────┐    ┌─────────────────────┐       │
│  │ 500 MoMo │───▶│  S3 Raw Bucket   │───▶│    AWS Lambda       │       │
│  │ Txns     │    │  (Lab 1)         │    │    run_pipeline()   │       │
│  └──────────┘    └──────────────────┘    │    (Lab 4)          │       │
│                                          └──────────┬──────────┘       │
│                                                     │                   │
│                      ┌──────────────────────────────┤                   │
│                       │                             │                   │
│                       ▼                             ▼                   │
│             ┌──────────────────┐         ┌──────────────────┐          │
│             │ S3 Processed     │         │   DynamoDB       │          │
│             │ (Lab 2 output)   │         │   (Lab 3)        │          │
│             └──────────────────┘         └──────────────────┘          │
│                                                     │                   │
│                                      ┌──────────────┤                   │
│                                      │              │                   │
│                                      ▼              ▼                   │
│                               ┌──────────┐   ┌──────────────┐          │
│                               │   SNS    │   │  CloudWatch  │          │
│                               │ Alerts   │   │  Monitoring  │          │
│                               └──────────┘   └──────────────┘          │
│                                                                         │
│                       ⏰ EventBridge: cron(0 22 * * ? *)               │
└─────────────────────────────────────────────────────────────────────────┘
```

## Skills You've Practiced

| Lab | Skill |
|-----|-------|
| Lab 1 | Uploading data to S3, working with boto3 |
| Lab 2 | Data cleaning with pandas: phones, duplicates, regions, amounts |
| Lab 3 | DynamoDB writes and queries, SNS alerting |
| Lab 4 | Full pipeline orchestration, logging, Lambda structure |

---

## 🚀 What to Explore Next

| Topic | AWS Service | Why Learn It |
|-------|------------|-------------|
| **Real-time fraud detection** | Amazon Kinesis | Process transactions as they happen |
| **Large-scale ETL (millions of rows)** | AWS Glue | Managed Spark for big data |
| **Complex pipeline orchestration** | AWS Step Functions | Multi-step workflows with retries |
| **SQL analytics on S3 data** | Amazon Athena | Query your Parquet files with SQL |
| **BI dashboards** | Amazon QuickSight | Visualize your MoMo data |
| **Infrastructure as Code** | AWS CloudFormation / CDK | Deploy all of this reproducibly |

## 📚 Resources

- **AWS Free Tier**: Create your free account at [aws.amazon.com/free](https://aws.amazon.com/free) — everything we built today fits in the free tier
- **boto3 documentation**: [boto3.amazonaws.com/v1/documentation](https://boto3.amazonaws.com/v1/documentation/api/latest/index.html)
- **moto documentation**: [docs.getmoto.org](https://docs.getmoto.org)
- **Ghana Fintech Community**: Search for *Ghana Fintech & Payments Forum* on LinkedIn
- **Bank of Ghana EMI Guidelines**: [bog.gov.gh](https://www.bog.gov.gh) → Payments & Currency → Electronic Money
- **Ghana Data Protection Commission**: [dataprotection.org.gh](https://dataprotection.org.gh)
- **GhIPSS**: [ghipss.net](https://www.ghipss.net) — Ghana's interbank payments backbone

---

> 🇬🇭 *"The engineers who understand data pipelines will be the ones building the infrastructure that powers the next MTN MoMo, the next Zeepay, the next GhIPSS. That engineer can be you. Keep building."*

---

**Workshop complete! 🎉 Well done.**